In [0]:
# Databricks Notebook: Control Table Generator (Batch Sources Only)

import yaml
from pyspark.sql import Row

# Load YAML config
config_path = "/dbfs/FileStore/configs/pipeline_config.yml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Validate batch sources only
def validate_source(source):
    required = ["name", "type", "mode", "bronze_path"]
    missing = [k for k in required if k not in source]
    if missing:
        raise ValueError(f"Missing required keys in {source.get('name', '?')}: {missing}")
    return source

# Filter for batch sources only
batch_sources = [Row(**validate_source(src)) for src in config["sources"] if src["mode"] == "batch"]

expected_fields = [
    "name", "type", "mode", "bronze_path",
    "base_url", "params", "incremental_field", 
    "base_path", "file_pattern", "kafka_topic"
]

def normalize_source_dict(src):
    return {field: src.get(field, None) for field in expected_fields}

# Now safely create a unified DataFrame
normalized_sources = [Row(**normalize_source_dict(src)) for src in config["sources"] if src["mode"] == "batch"]
control_df = spark.createDataFrame(normalized_sources)

# Register as a Delta table
control_df.write.format("delta").mode("overwrite").saveAsTable("control.pipeline_config_batch")

display(control_df)


name,type,mode,bronze_path,base_url,params,incremental_field,base_path,file_pattern,kafka_topic
healthcare_api,api,batch,/FileStore/bronze/bronze_healthcare_patients,https://hapi.fhir.org/baseR4/Patient,Map(_count -> 100),lastUpdated,null,null,patient_data
gov_pdfs,file,batch,/FileStore/bronze/bronze_pdfs,null,null,null,/FileStore/raw_pdfs,*.pdf,gov_pdfs
